In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import sys

from model_binary_resnet import get_resnet18_binary_model
from model_brand_resnet import get_resnet18_brand_model


In [ ]:

#  경로 설정
binary_model_path = "best_binary_model.pth"
brand_model_path  = "best_brand_model.pth"

binary_classes = ['food', 'not_food']
brand_classes = ['burgerking', 'lotteria', 'shakeshack', 'subway', 'eggdrop', 'paikdabang', 'johnnys', 'momstouch', 'puradak']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

#  모델 로딩
binary_model = get_resnet18_binary_model(pretrained=False).to(device)
binary_model.load_state_dict(torch.load(binary_model_path, map_location=device))
binary_model.eval()

brand_model = get_resnet18_brand_model(num_classes=len(brand_classes), pretrained=False).to(device)
brand_model.load_state_dict(torch.load(brand_model_path, map_location=device))
brand_model.eval()


def load_image(input_path_or_url):
    try:
        if input_path_or_url.startswith("http"):
            response = requests.get(input_path_or_url)
            image = Image.open(BytesIO(response.content)).convert("RGB")
        else:
            image = Image.open(input_path_or_url).convert("RGB")
        return image
    except Exception as e:
        print(f" 이미지 불러오기 실패: {e}")
        return None


def predict(input_path_or_url):
    image = load_image(input_path_or_url)
    if image is None:
        return

    input_tensor = transform(image).unsqueeze(0).to(device)

    #  이진 분류 (food / not_food)
    with torch.no_grad():
        output_bin = binary_model(input_tensor)
        prob_bin = F.softmax(output_bin, dim=1)
        pred_bin = prob_bin.argmax(dim=1).item()
        label_bin = binary_classes[pred_bin]
        confidence_bin = prob_bin[0][pred_bin].item()

    print(f" [1차 분류] → {label_bin} ({confidence_bin*100:.2f}%)")

    if label_bin == 'not_food':
        plt.imshow(image)
        plt.title(f"[NOT_FOOD] ({confidence_bin*100:.2f}%)")
        plt.axis("off")
        plt.show()
        return

    #  브랜드 분류
    with torch.no_grad():
        output_brand = brand_model(input_tensor)
        prob_brand = F.softmax(output_brand, dim=1)
        top3_probs, top3_indices = torch.topk(prob_brand, 3)

    print(f" [2차 분류] 브랜드 Top-3:")
    for i in range(3):
        brand = brand_classes[top3_indices[0][i]]
        prob = top3_probs[0][i].item() * 100
        print(f"{i+1}. {brand}: {prob:.2f}%")

    # 이미지 시각화
    top1_brand = brand_classes[top3_indices[0][0]]
    top1_prob = top3_probs[0][0].item() * 100
    plt.imshow(image)
    plt.title(f"{top1_brand} ({top1_prob:.2f}%)", fontsize=12)
    plt.axis("off")
    plt.show()


if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("사용법: python predict.py [이미지경로 또는 이미지URL]")
    else:
        input_path = sys.argv[1]
        predict(input_path)
